# 12 Publication-Quality Figures and Tables

Create paper-ready figures and tables from the report artifacts generated by earlier notebooks. Outputs are written as vector PDFs/SVGs and high-resolution PNGs under reports/publication/.


In [ ]:
from pathlib import Path
import json

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_recall_curve, average_precision_score

mpl.use('Agg')


In [ ]:
REPO_ROOT = Path('..').resolve()
REPORT_DIR = REPO_ROOT / 'reports'
PUBLICATION_DIR = REPORT_DIR / 'publication'
FIGURE_DIR = PUBLICATION_DIR / 'figures'
TABLE_DIR = PUBLICATION_DIR / 'tables'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_SUMMARY_PATH = REPORT_DIR / 'manifests' / 'turning_split_summary.csv'
BASELINE_METRICS_PATH = REPORT_DIR / 'tables' / 'metrics_turning_baselines.csv'
BASELINE_METRICS_CI_PATH = REPORT_DIR / 'tables' / 'metrics_turning_baselines_with_ci.csv'
BASELINE_SCORES_PATH = REPORT_DIR / 'baselines' / 'baseline_scores_turning.csv'
BASELINE_THRESHOLDS_PATH = REPORT_DIR / 'thresholds' / 'baseline_thresholds_turning.json'
AE_SCORES_PATH = REPORT_DIR / 'scores' / 'ae_scores_turning.csv'
AE_THRESHOLDS_PATH = REPORT_DIR / 'thresholds' / 'ae_thresholds_turning.json'

METHOD_LABELS = {
    'one_class_svm_features': 'One-class SVM',
    'isolation_forest_features': 'Isolation forest',
    'pca_image_reconstruction': 'PCA reconstruction',
}
METRIC_LABELS = {
    'pr_auc': 'PR-AUC',
    'f1': 'F1-score',
    'precision': 'Precision',
    'recall': 'Recall',
}

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 600,
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 9,
    'legend.fontsize': 7,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})


In [ ]:
def save_figure(fig, stem: str) -> None:
    for ext in ['pdf', 'svg', 'png']:
        fig.savefig(FIGURE_DIR / f'{stem}.{ext}', bbox_inches='tight')
    plt.close(fig)

def interpolated_precision_recall(y_true, scores):
    precision, recall, _ = precision_recall_curve(y_true, scores)
    order = np.argsort(recall)
    recall = recall[order]
    precision = precision[order]
    recall_unique = np.unique(recall)
    precision_unique = np.array([precision[recall == value].max() for value in recall_unique])
    precision_envelope = np.maximum.accumulate(precision_unique[::-1])[::-1]
    return recall_unique, precision_envelope

split_summary = pd.read_csv(SPLIT_SUMMARY_PATH)
baseline_metrics = pd.read_csv(BASELINE_METRICS_CI_PATH if BASELINE_METRICS_CI_PATH.exists() else BASELINE_METRICS_PATH)
baseline_scores = pd.read_csv(BASELINE_SCORES_PATH)
with BASELINE_THRESHOLDS_PATH.open() as f:
    baseline_thresholds = json.load(f)

baseline_metrics['method_label'] = baseline_metrics['method'].map(METHOD_LABELS).fillna(baseline_metrics['method'])
baseline_scores['method_label'] = baseline_scores['method'].map(METHOD_LABELS).fillna(baseline_scores['method'])


## Dataset Split Figure and Table


In [ ]:
split_pivot = split_summary.pivot_table(index='split', columns='label', values='n', fill_value=0)
split_pivot = split_pivot.reindex(['train', 'validation', 'test']).fillna(0)
split_pivot = split_pivot.rename(columns={'no_chatter': 'Nominal', 'chatter': 'Chatter'})

fig, ax = plt.subplots(figsize=(3.35, 2.1))
bottom = np.zeros(len(split_pivot))
colors = {'Nominal': '#4C78A8', 'Chatter': '#F58518'}
for label in ['Nominal', 'Chatter']:
    values = split_pivot[label].to_numpy() if label in split_pivot else np.zeros(len(split_pivot))
    ax.bar(split_pivot.index, values, bottom=bottom, label=label, color=colors[label], width=0.62)
    for x, y0, value in zip(range(len(values)), bottom, values):
        if value > 0:
            ax.text(x, y0 + value / 2, f'{int(value)}', ha='center', va='center', color='white', fontsize=7)
    bottom += values
ax.set_ylabel('Number of windows')
ax.set_xlabel('Split')
ax.legend(frameon=False, loc='upper right')
ax.set_title('Frozen turning-dataset split')
ax.grid(axis='y', color='0.9', linewidth=0.8)
save_figure(fig, 'turning_split_counts')

split_pivot.to_csv(TABLE_DIR / 'turning_split_counts_publication.csv')
split_pivot.to_latex(TABLE_DIR / 'turning_split_counts_publication.tex')
split_pivot


## Baseline Metric Figure and Table


In [ ]:
metric_order = ['pr_auc', 'f1', 'precision', 'recall']
plot_table = baseline_metrics.set_index('method_label').loc[[METHOD_LABELS[m] for m in METHOD_LABELS], metric_order]

fig, ax = plt.subplots(figsize=(6.8, 2.45))
x = np.arange(len(plot_table.index))
bar_width = 0.18
palette = ['#4C78A8', '#F58518', '#54A24B', '#B279A2']
for i, metric in enumerate(metric_order):
    values = plot_table[metric].to_numpy()
    offsets = x + (i - 1.5) * bar_width
    ax.bar(offsets, values, width=bar_width, label=METRIC_LABELS[metric], color=palette[i])
ax.set_xticks(x)
ax.set_xticklabels(plot_table.index, rotation=0)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Baseline anomaly-detection performance on held-out test split')
ax.legend(frameon=False, ncol=4, loc='lower center', bbox_to_anchor=(0.5, 1.02))
ax.grid(axis='y', color='0.9', linewidth=0.8)
save_figure(fig, 'baseline_metric_panel')

publication_metrics = baseline_metrics[['method_label', 'pr_auc', 'f1', 'precision', 'recall', 'tn', 'fp', 'fn', 'tp']].copy()
publication_metrics = publication_metrics.rename(columns={'method_label': 'Method', 'pr_auc': 'PR-AUC', 'f1': 'F1-score', 'precision': 'Precision', 'recall': 'Recall', 'tn': 'TN', 'fp': 'FP', 'fn': 'FN', 'tp': 'TP'})
for source, label in [('pr_auc', 'PR-AUC'), ('f1', 'F1-score'), ('precision', 'Precision'), ('recall', 'Recall')]:
    low = f'{source}_ci_low'
    high = f'{source}_ci_high'
    if low in baseline_metrics.columns and high in baseline_metrics.columns:
        publication_metrics[f'{label} 95% CI'] = [f'[{lo:.3f}, {hi:.3f}]' for lo, hi in zip(baseline_metrics[low], baseline_metrics[high])]
publication_metrics.to_csv(TABLE_DIR / 'baseline_metrics_publication.csv', index=False)
publication_metrics.to_latex(TABLE_DIR / 'baseline_metrics_publication.tex', index=False, float_format='%.3f')
publication_metrics


## Precision-Recall Curves


In [ ]:
fig, ax = plt.subplots(figsize=(3.35, 2.65))
for method, group in baseline_scores[baseline_scores['split'] == 'test'].groupby('method'):
    y_true = group['target'].to_numpy()
    scores = group['score_value'].to_numpy()
    recall, precision = interpolated_precision_recall(y_true, scores)
    ap = average_precision_score(y_true, scores)
    ax.step(recall, precision, where='post', linewidth=1.8, label=f'{METHOD_LABELS.get(method, method)} ({ap:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.set_title('Interpolated precision-recall curves')
ax.grid(color='0.9', linewidth=0.8)
ax.legend(frameon=False, loc='lower left')
save_figure(fig, 'baseline_precision_recall_curves')


## Confusion Matrices


In [ ]:
methods = list(METHOD_LABELS)
fig, axes = plt.subplots(1, len(methods), figsize=(6.8, 2.25), constrained_layout=True)
if len(methods) == 1:
    axes = [axes]
for ax, method in zip(axes, methods):
    group = baseline_scores[(baseline_scores['split'] == 'test') & (baseline_scores['method'] == method)]
    threshold = baseline_thresholds[method]['threshold']
    y_true = group['target'].to_numpy()
    y_pred = (group['score_value'].to_numpy() >= threshold).astype(int)
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    ax.imshow(matrix, cmap='Blues', vmin=0)
    for (row, col), value in np.ndenumerate(matrix):
        ax.text(col, row, str(value), ha='center', va='center', color='black', fontsize=9)
    ax.set_xticks([0, 1], labels=['Nominal', 'Chatter'], rotation=35, ha='right')
    ax.set_yticks([0, 1], labels=['Nominal', 'Chatter'])
    ax.set_xlabel('Predicted')
    ax.set_title(METHOD_LABELS.get(method, method))
axes[0].set_ylabel('True')
save_figure(fig, 'baseline_confusion_matrices')


## CNN-AE Architecture Figure and Table

These publication artifacts document the CNN autoencoder architecture used in notebook 04. They do not require TensorFlow and can be regenerated from the model specification in the notebook.


In [ ]:
ae_layers = pd.DataFrame([
    {'stage': 'Input', 'layer': 'Input image', 'type': 'Input', 'kernel': '', 'stride_pool': '', 'padding': '', 'activation': '', 'output_shape': '100 x 150 x 3', 'params': 0},
    {'stage': 'Encoder', 'layer': 'Conv2D-4', 'type': 'Conv2D', 'kernel': '3 x 3', 'stride_pool': '1 x 1', 'padding': 'same', 'activation': 'ReLU', 'output_shape': '100 x 150 x 4', 'params': 112},
    {'stage': 'Encoder', 'layer': 'MaxPool', 'type': 'MaxPooling2D', 'kernel': '', 'stride_pool': '2 x 2', 'padding': 'same', 'activation': '', 'output_shape': '50 x 75 x 4', 'params': 0},
    {'stage': 'Encoder', 'layer': 'Conv2D-8', 'type': 'Conv2D', 'kernel': '3 x 3', 'stride_pool': '1 x 1', 'padding': 'same', 'activation': 'ReLU', 'output_shape': '50 x 75 x 8', 'params': 296},
    {'stage': 'Encoder', 'layer': 'MaxPool', 'type': 'MaxPooling2D', 'kernel': '', 'stride_pool': '2 x 3', 'padding': 'same', 'activation': '', 'output_shape': '25 x 25 x 8', 'params': 0},
    {'stage': 'Encoder', 'layer': 'Conv2D-12', 'type': 'Conv2D', 'kernel': '3 x 3', 'stride_pool': '1 x 1', 'padding': 'same', 'activation': 'ReLU', 'output_shape': '25 x 25 x 12', 'params': 876},
    {'stage': 'Bottleneck', 'layer': 'Flatten', 'type': 'Flatten', 'kernel': '', 'stride_pool': '', 'padding': '', 'activation': '', 'output_shape': '7500', 'params': 0},
    {'stage': 'Bottleneck', 'layer': 'Dense-16', 'type': 'Dense', 'kernel': '', 'stride_pool': '', 'padding': '', 'activation': 'ReLU', 'output_shape': '16', 'params': 120016},
    {'stage': 'Decoder', 'layer': 'Dense-7500', 'type': 'Dense', 'kernel': '', 'stride_pool': '', 'padding': '', 'activation': 'ReLU', 'output_shape': '7500', 'params': 127500},
    {'stage': 'Decoder', 'layer': 'Reshape', 'type': 'Reshape', 'kernel': '', 'stride_pool': '', 'padding': '', 'activation': '', 'output_shape': '25 x 25 x 12', 'params': 0},
    {'stage': 'Decoder', 'layer': 'Conv2D-12', 'type': 'Conv2D', 'kernel': '3 x 3', 'stride_pool': '1 x 1', 'padding': 'same', 'activation': 'ReLU', 'output_shape': '25 x 25 x 12', 'params': 1308},
    {'stage': 'Decoder', 'layer': 'UpSampling', 'type': 'UpSampling2D', 'kernel': '', 'stride_pool': '2 x 3', 'padding': '', 'activation': '', 'output_shape': '50 x 75 x 12', 'params': 0},
    {'stage': 'Decoder', 'layer': 'Conv2D-8', 'type': 'Conv2D', 'kernel': '3 x 3', 'stride_pool': '1 x 1', 'padding': 'same', 'activation': 'ReLU', 'output_shape': '50 x 75 x 8', 'params': 872},
    {'stage': 'Decoder', 'layer': 'UpSampling', 'type': 'UpSampling2D', 'kernel': '', 'stride_pool': '2 x 2', 'padding': '', 'activation': '', 'output_shape': '100 x 150 x 8', 'params': 0},
    {'stage': 'Decoder', 'layer': 'Conv2D-4', 'type': 'Conv2D', 'kernel': '3 x 3', 'stride_pool': '1 x 1', 'padding': 'same', 'activation': 'ReLU', 'output_shape': '100 x 150 x 4', 'params': 292},
    {'stage': 'Output', 'layer': 'Conv2D-3', 'type': 'Conv2D', 'kernel': '3 x 3', 'stride_pool': '1 x 1', 'padding': 'same', 'activation': 'Sigmoid', 'output_shape': '100 x 150 x 3', 'params': 111},
])
ae_layers.to_csv(TABLE_DIR / 'cnn_ae_architecture_publication.csv', index=False)
ae_layers.to_latex(TABLE_DIR / 'cnn_ae_architecture_publication.tex', index=False)
print('Total parameters:', int(ae_layers['params'].sum()))
ae_layers


In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_block(ax, x, y, w, h, title, subtitle, color):
    patch = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02,rounding_size=0.025',
                           linewidth=0.9, edgecolor='0.25', facecolor=color)
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h * 0.62, title, ha='center', va='center', fontsize=8, weight='bold')
    ax.text(x + w / 2, y + h * 0.32, subtitle, ha='center', va='center', fontsize=6.8)

def draw_arrow(ax, start, end):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle='-|>', mutation_scale=9, linewidth=0.9, color='0.25'))

fig, ax = plt.subplots(figsize=(7.0, 2.45))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

blocks = [
    (0.02, 0.33, 0.12, 0.34, 'Input', '100 x 150 x 3', '#E8EEF7'),
    (0.18, 0.33, 0.13, 0.34, 'Encoder', 'Conv 4/8/12\nPool 2x2, 2x3', '#D7E8D4'),
    (0.36, 0.33, 0.13, 0.34, 'Latent', 'Dense 16', '#F6E2B8'),
    (0.54, 0.33, 0.13, 0.34, 'Decoder', 'Dense + reshape\nUpsample 2x3, 2x2', '#D7E8D4'),
    (0.72, 0.33, 0.12, 0.34, 'Output', '100 x 150 x 3', '#E8EEF7'),
    (0.88, 0.33, 0.10, 0.34, 'Error', 'x - xhat', '#F1C7C2'),
]
for block in blocks:
    draw_block(ax, *block)
for x0, x1 in [(0.14, 0.18), (0.31, 0.36), (0.49, 0.54), (0.67, 0.72), (0.84, 0.88)]:
    draw_arrow(ax, (x0, 0.50), (x1, 0.50))

ax.text(0.50, 0.88, 'CNN autoencoder with 16-dimensional bottleneck', ha='center', fontsize=9, weight='bold')
ax.text(0.50, 0.12, 'Training uses nominal spectrograms only; anomaly scores are derived from reconstruction error.', ha='center', fontsize=7)
save_figure(fig, 'cnn_ae_architecture')


## CNN-AE Scoring Workflow Figure


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 2.6))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

workflow_blocks = [
    (0.02, 0.52, 0.14, 0.28, 'Spectrogram', 'X/Y/Z RGB\n150 x 100', '#E8EEF7'),
    (0.22, 0.52, 0.15, 0.28, 'CNN-AE', 'reconstructs\ninput image', '#D7E8D4'),
    (0.43, 0.52, 0.15, 0.28, 'Error Map', 'squared / absolute\npixel error', '#F1C7C2'),
    (0.66, 0.66, 0.14, 0.22, 'Global Scores', 'MSE, MAE', '#F6E2B8'),
    (0.66, 0.36, 0.14, 0.22, 'VER Scores', 'max segment\ntop-k segments', '#F6E2B8'),
    (0.86, 0.52, 0.12, 0.28, 'Decision', 'frozen validation\nthreshold', '#E3D7F0'),
]
for block in workflow_blocks:
    draw_block(ax, *block)
draw_arrow(ax, (0.16, 0.66), (0.22, 0.66))
draw_arrow(ax, (0.37, 0.66), (0.43, 0.66))
draw_arrow(ax, (0.58, 0.66), (0.66, 0.77))
draw_arrow(ax, (0.58, 0.62), (0.66, 0.47))
draw_arrow(ax, (0.80, 0.77), (0.86, 0.68))
draw_arrow(ax, (0.80, 0.47), (0.86, 0.60))
ax.text(0.50, 0.93, 'CNN-AE anomaly-scoring workflow', ha='center', fontsize=9, weight='bold')
ax.text(0.50, 0.14, 'Thresholds are selected on validation data and then held fixed for final test evaluation.', ha='center', fontsize=7)
save_figure(fig, 'cnn_ae_scoring_workflow')


## CNN-AE Precision-Recall Curves and Confusion Matrices

These figures are generated when `reports/scores/ae_scores_turning.csv` and `reports/thresholds/ae_thresholds_turning.json` exist. They are produced by notebook 05 after TensorFlow can load the trained CNN-AE model.


In [ ]:
AE_SCORE_LABELS = {
    'global_mse': 'Global MSE',
    'global_mae': 'Global MAE',
    'ver_max': 'VER max',
    'ver_topk': 'VER top-k',
}

if AE_SCORES_PATH.exists() and AE_THRESHOLDS_PATH.exists():
    ae_scores = pd.read_csv(AE_SCORES_PATH)
    with AE_THRESHOLDS_PATH.open() as f:
        ae_thresholds = json.load(f)
    ae_test = ae_scores[ae_scores['split'] == 'test'].copy()
    ae_test['target'] = (ae_test['label'] == 'chatter').astype(int)

    fig, ax = plt.subplots(figsize=(3.35, 2.65))
    ae_metric_rows = []
    for score_name, label in AE_SCORE_LABELS.items():
        y_true = ae_test['target'].to_numpy()
        score_values = ae_test[score_name].to_numpy()
        recall, precision = interpolated_precision_recall(y_true, score_values)
        ap = average_precision_score(y_true, score_values)
        ax.step(recall, precision, where='post', linewidth=1.8, label=f'{label} ({ap:.3f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.02)
    ax.set_title('CNN-AE interpolated precision-recall curves')
    ax.grid(color='0.9', linewidth=0.8)
    ax.legend(frameon=False, loc='lower left')
    save_figure(fig, 'cnn_ae_precision_recall_curves')

    score_names = list(AE_SCORE_LABELS)
    fig, axes = plt.subplots(2, 2, figsize=(5.0, 4.3), constrained_layout=True)
    for ax, score_name in zip(axes.ravel(), score_names):
        threshold = float(ae_thresholds[score_name]['threshold'])
        y_true = ae_test['target'].to_numpy()
        y_pred = (ae_test[score_name].to_numpy() >= threshold).astype(int)
        matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
        ax.imshow(matrix, cmap='Blues', vmin=0)
        for (row, col), value in np.ndenumerate(matrix):
            ax.text(col, row, str(value), ha='center', va='center', color='black', fontsize=9)
        ax.set_xticks([0, 1], labels=['Nominal', 'Chatter'], rotation=35, ha='right')
        ax.set_yticks([0, 1], labels=['Nominal', 'Chatter'])
        ax.set_xlabel('Predicted')
        ax.set_title(AE_SCORE_LABELS[score_name])
        if ax in axes[:, 0]:
            ax.set_ylabel('True')
        tn, fp, fn, tp = matrix.ravel()
        ae_metric_rows.append({'score': score_name, 'threshold': threshold, 'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp})
    save_figure(fig, 'cnn_ae_confusion_matrices')

    ae_confusion_table = pd.DataFrame(ae_metric_rows)
    ae_confusion_table.to_csv(TABLE_DIR / 'cnn_ae_confusion_matrices_publication.csv', index=False)
    ae_confusion_table.to_latex(TABLE_DIR / 'cnn_ae_confusion_matrices_publication.tex', index=False)
    (PUBLICATION_DIR / 'cnn_ae_pr_confusion_pending.md').unlink(missing_ok=True)
    print('Wrote CNN-AE PR curves and confusion matrices.')
else:
    pending_path = PUBLICATION_DIR / 'cnn_ae_pr_confusion_pending.md'
    pending_path.write_text(
        '# CNN-AE PR Curves and Confusion Matrices Pending\n\n'
        'The CNN-AE PR curves and confusion matrices require `reports/scores/ae_scores_turning.csv` '
        'and `reports/thresholds/ae_thresholds_turning.json`. Run notebook 05 after TensorFlow is available '
        'to load the trained `.keras` model and produce those score artifacts.\n'
    )
    print(f'CNN-AE score artifacts not found; wrote {pending_path}')


## Artifact Index


In [ ]:
artifacts = sorted([p.relative_to(REPO_ROOT).as_posix() for p in PUBLICATION_DIR.rglob('*') if p.is_file()])
artifact_index = pd.DataFrame({'artifact': artifacts})
artifact_index.to_csv(PUBLICATION_DIR / 'artifact_index.csv', index=False)
artifact_index
